In [2]:
import os

print(os.getcwd())

c:\Users\jinso.JINSUWON\Paper_Study_Note\PDDL_GBF_HFF


In [4]:
!pip install pyperplan

In [5]:
domain_pddl = """
(define (domain BLOCKS)
  (:requirements :strips :typing)
  (:types block)

  (:predicates
    (on ?x - block ?y - block)
    (ontable ?x - block)
    (clear ?x - block)
    (holding ?x - block)
    (handempty)
  )

  (:action pick-up
    :parameters (?x - block)
    :precondition
      (and
        (clear ?x)
        (ontable ?x)
        (handempty)
      )
    :effect
      (and
        (not (clear ?x))
        (not (ontable ?x))
        (not (handempty))
        (holding ?x)
      )
  )

  (:action put-down
    :parameters (?x - block)
    :precondition
      (holding ?x)
    :effect
      (and
        (not (holding ?x))
        (handempty)
        (ontable ?x)
        (clear ?x)
      )
  )

  (:action stack
    :parameters (?x - block ?y - block)
    :precondition
      (and
        (holding ?x)
        (clear ?y)
      )
    :effect
      (and
        (not (holding ?x))
        (handempty)
        (clear ?x)
        (not (clear ?y))
        (on ?x ?y)
      )
  )

  (:action unstack
    :parameters (?x - block ?y - block)
    :precondition
      (and
        (on ?x ?y)
        (clear ?x)
        (handempty)
      )
    :effect
      (and
        (holding ?x)
        (clear ?y)
        (not (clear ?x))
        (not (handempty))
        (not (on ?x ?y))
      )
  )
)
"""

with open("domain.pddl", "w") as f:
    f.write(domain_pddl)

In [6]:
problem_pddl = """
(define (problem BLOCK-6)
  (:domain BLOCKS)

  (:objects A B C D E F - block)

  (:init
    (handempty)
    (clear A)
    (clear D)
    (clear F)
    (ontable C)
    (ontable D)
    (ontable B)
    (on A E)
    (on E C)
    (on F B)
  )

  (:goal
    (and
      (on B C)
      (on C F)
      (on F A)
      (on D E)
    )
  )
)
"""

with open("problem.pddl", "w") as f:
    f.write(problem_pddl)

In [7]:
!pyperplan -H hff -s gbf domain.pddl problem.pddl

2026-06-12 13:01:32,111 INFO     using search: greedy_best_first_search
2026-06-12 13:01:32,111 INFO     using heuristic: hFFHeuristic
2026-06-12 13:01:32,111 INFO     Parsing Domain c:\Users\jinso.JINSUWON\Paper_Study_Note\PDDL_GBF_HFF\domain.pddl
2026-06-12 13:01:32,113 INFO     Parsing Problem c:\Users\jinso.JINSUWON\Paper_Study_Note\PDDL_GBF_HFF\problem.pddl
2026-06-12 13:01:32,114 INFO     5 Predicates parsed
2026-06-12 13:01:32,114 INFO     4 Actions parsed
2026-06-12 13:01:32,114 INFO     6 Objects parsed
2026-06-12 13:01:32,114 INFO     0 Constants parsed
2026-06-12 13:01:32,115 INFO     Grounding start: block-6
2026-06-12 13:01:32,115 INFO     Relevance analysis removed 0 facts
2026-06-12 13:01:32,115 INFO     Grounding end: block-6
2026-06-12 13:01:32,115 INFO     55 Variables created
2026-06-12 13:01:32,115 INFO     84 Operators created
2026-06-12 13:01:32,116 INFO     Search start: block-6
2026-06-12 13:01:32,116 INFO     Initial h value: 10.000000
2026-06-12 13:01:32,136 I

In [8]:
!type problem.pddl.sol

������ ������ ã�� �� �����ϴ�.


In [9]:
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.patches as patches

stacks = [["C", "E", "A"], ["D"], ["B", "F"]]
holding = None

plan_file = Path("problem.pddl.soln")
out_dir = Path("gbf_hff_blockworld_steps")
out_dir.mkdir(exist_ok=True)

plan = []
with open(plan_file, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            line = line.replace("(", "").replace(")", "")
            plan.append(tuple(line.split()))

def normalize(block):
    return block.upper()

def find_stack(block):
    block = normalize(block)
    for i, stack in enumerate(stacks):
        if block in stack:
            return i
    return None

def remove_empty_stacks():
    while [] in stacks:
        stacks.remove([])

def draw_state(step, action=None):
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.set_xlim(0, max(5, len(stacks) + 2))
    ax.set_ylim(0, 7)
    ax.axis("off")

    block_w = 0.8
    block_h = 0.45

    ax.text(0.1, 6.5, f"Step {step}", fontsize=14)
    if action:
        ax.text(0.1, 6.1, "Action: " + " ".join(action), fontsize=11)

    for i, stack in enumerate(stacks):
        x = i + 1
        ax.plot([x - 0.5, x + 0.5], [0.5, 0.5])

        for j, block in enumerate(stack):
            y = 0.6 + j * block_h
            rect = patches.Rectangle(
                (x - block_w / 2, y),
                block_w,
                block_h,
                fill=False
            )
            ax.add_patch(rect)
            ax.text(
                x,
                y + block_h / 2,
                block,
                ha="center",
                va="center",
                fontsize=12
            )

    if holding:
        ax.text(len(stacks) + 1, 5.5, f"Holding: {holding}", fontsize=12)

    plt.savefig(out_dir / f"gbf_hff_step_{step:02d}.png", bbox_inches="tight")
    plt.close()

def apply_action(action):
    global holding

    op = action[0]
    args = [normalize(x) for x in action[1:]]

    if op == "unstack":
        x, y = args
        idx = find_stack(x)
        stacks[idx].pop()
        holding = x

    elif op == "put-down":
        x = args[0]
        stacks.append([x])
        holding = None
        remove_empty_stacks()

    elif op == "pick-up":
        x = args[0]
        idx = find_stack(x)
        stacks[idx].pop()
        holding = x
        remove_empty_stacks()

    elif op == "stack":
        x, y = args
        idx = find_stack(y)
        stacks[idx].append(x)
        holding = None

draw_state(0)

for step, action in enumerate(plan, start=1):
    apply_action(action)
    draw_state(step, action)

print(f"Plan length: {len(plan)}")
print(f"Saved images to: {out_dir}")

Plan length: 16
Saved images to: gbf_hff_blockworld_steps
